# Tesis: Prototipo de Sistema Integrado IoT–PID–IA para Papa bajo Invernadero
## Fase 2: Identificación del Sistema (FOPDT), Sintonización Clásica y Optimización con Optuna

**Ubicación:** La Paz, Bolivia (Altitud ~3600 msnm, clima frío nocturno).
**Objetivos de esta fase:**
1. Cargar y procesar los datos experimentales de la curva de reacción al escalón.
2. Ajustar un modelo de Primer Orden con Tiempo Muerto (**FOPDT**: $G(s) = \frac{K e^{-\theta s}}{\tau s + 1}$).
3. Calcular ganancias analíticas mediante **Ziegler-Nichols**, **Cohen-Coon** e **IMC**.
4. Simular la dinámica en lazo cerrado y calcular métricas cuantitativas ($t_r, t_s, M_p, e_{ss}, IAE, ITAE$).
5. Optimizar ganancias globales mediante búsqueda bayesiana con **Optuna** (TPE Sampler).
6. Demostrar la superioridad del control PID continuo frente al control On/Off convencional con histéresis.

In [ ]:
# ==============================================================================
# Celda 1: Instalación de Dependencias (Google Colab / Entorno Local)
# ==============================================================================
# Si ejecutas en Google Colab, descomenta las siguientes líneas:
# !pip install -q optuna scipy pandas matplotlib seaborn

import os
import math
import numpy as np
import pandas as pd
import scipy.optimize as opt
import matplotlib.pyplot as plt
import optuna

# Silenciar logs verbosos de Optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Fijar semillas para reproducibilidad académica
np.random.seed(42)

# Configuración gráfica con formato de publicación IEEE/APA (alta resolución)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.linestyle'] = ':'
plt.rcParams['grid.alpha'] = 0.6
print("[✔] Entorno inicializado y librerías importadas con éxito.")

### Montaje opcional de Google Drive en Google Colab
Si deseas persistir las figuras y tablas directamente en tu Google Drive, ejecuta la siguiente celda:

In [ ]:
# ==============================================================================
# Celda 2: Montaje de Google Drive (Opcional en Colab)
# ==============================================================================
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUTA_BASE = '/content/drive/MyDrive/Tesis_IOT_Papa'
    print("[✔] Google Drive montado en /content/drive")
except Exception:
    RUTA_BASE = '.'
    print("[i] Ejecutando en entorno local o sin Google Drive montado.")

## 1. Carga y Visualización de la Curva de Reacción Experimental
La prueba consiste en un escalón del $70\%$ en la señal de control del calefactor cerámico ($u(t) = 70\%$) a partir de $t = 120\text{ s}$, partiendo del reposo térmico nocturno en La Paz ($8.5^\circ\text{C}$).

In [ ]:
# ==============================================================================
# Celda 3: Carga del Dataset de Escalón Térmico
# ==============================================================================
csv_path = os.path.join(RUTA_BASE, 'data/synthetic/prueba_escalon_fopdt.csv')

# Si el archivo no existe (por ejemplo, en una sesión limpia de Colab), generarlo:
if not os.path.exists(csv_path):
    print("[*] Generando dataset sintético de la dinámica térmica del invernadero...")
    dt_gen = 1.0
    t_arr = np.arange(0, 1800, dt_gen)
    u_arr = np.where(t_arr >= 120.0, 70.0, 0.0)
    y_arr = np.zeros_like(t_arr)
    y_arr[0] = 8.5
    for i in range(1, len(t_arr)):
        u_ef = 70.0 if (t_arr[i] - 35.0) >= 120.0 else 0.0
        dy = ( - (y_arr[i-1] - 8.5) + (0.165 * u_ef) ) / 240.0 * dt_gen
        y_arr[i] = y_arr[i-1] + dy
    ruido = np.random.normal(0, 0.08, len(t_arr)) + 0.15 * np.sin(2 * np.pi * t_arr / 600.0)
    df_gen = pd.DataFrame({'tiempo_segundos': t_arr, 'control_u_pct': u_arr, 'temperatura_medida_c': np.round(y_arr + ruido, 2)})
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    df_gen.to_csv(csv_path, index=False)

df = pd.read_csv(csv_path)
t_exp = df['tiempo_segundos'].values
u_exp = df['control_u_pct'].values
y_exp = df['temperatura_medida_c'].values

print(f"[✔] Datos cargados: {len(df)} muestras | Duración: {t_exp[-1]/60:.1f} minutos")
print(df.head())

## 2. Identificación Paramétrica del Modelo FOPDT

La función de transferencia del modelo FOPDT (First Order Plus Dead Time) en el dominio de Laplace es:
$$
G(s) = \frac{Y(s)}{U(s)} = \frac{K e^{-\theta s}}{\tau s + 1}
$$

Su respuesta temporal ante una entrada escalón de magnitud $\Delta u$ en el instante $t_{step}$ es:
$$
y(t) = y_0 + K \cdot \Delta u \cdot \left( 1 - e^{-\frac{t - (t_{step} + \theta)}{\tau}} \right) \cdot \mathbb{I}(t \ge t_{step} + \theta)
$$

Ajustamos los parámetros $K, \tau, \theta$ mediante **Regresión No Lineal por Mínimos Cuadrados** (`scipy.optimize.curve_fit`).

In [ ]:
# ==============================================================================
# Celda 4: Ajuste No Lineal FOPDT
# ==============================================================================
t_step = 120.0
delta_u = 70.0
y0 = float(np.mean(y_exp[t_exp < t_step]))
y_inf = float(np.mean(y_exp[t_exp >= 1500]))
delta_y = y_inf - y0
K_init = delta_y / delta_u

def modelo_fopdt(t, K, tau, theta):
    y = np.full_like(t, y0, dtype=float)
    idx = t >= (t_step + theta)
    t_ef = t[idx] - (t_step + theta)
    y[idx] = y0 + K * delta_u * (1.0 - np.exp(-t_ef / tau))
    return y

popt, pcov = opt.curve_fit(
    modelo_fopdt, t_exp, y_exp,
    p0=[K_init, 200.0, 30.0],
    bounds=([0.05, 50.0, 5.0], [0.5, 600.0, 100.0])
)
K_opt, tau_opt, theta_opt = popt
y_pred = modelo_fopdt(t_exp, *popt)

# Métricas de bondad de ajuste
ss_res = np.sum((y_exp - y_pred)**2)
ss_tot = np.sum((y_exp - np.mean(y_exp))**2)
r2 = 1.0 - (ss_res / ss_tot)
rmse = np.sqrt(np.mean((y_exp - y_pred)**2))

print("=" * 50)
print("PARÁMETROS FOPDT IDENTIFICADOS")
print("=" * 50)
print(f"Ganancia Estática (K)   : {K_opt:.4f} °C / %")
print(f"Constante de Tiempo (τ) : {tau_opt:.2f} s ({tau_opt/60:.2f} min)")
print(f"Tiempo Muerto (θ)       : {theta_opt:.2f} s")
print(f"Relación de Retardo θ/τ : {theta_opt/tau_opt:.3f}")
print(f"Coeficiente R²          : {r2:.4f}")
print(f"RMSE                    : {rmse:.4f} °C")
print("=" * 50)

In [ ]:
# ==============================================================================
# Celda 5: Gráfico de Identificación FOPDT para la Tesis
# ==============================================================================
plt.figure(figsize=(9, 4.8), dpi=300)
plt.plot(t_exp / 60.0, y_exp, color='#4A5568', alpha=0.6, label='Sensor SHT31 (Datos Experimentales)', linewidth=1.2)
plt.plot(t_exp / 60.0, y_pred, color='#E53E3E', linewidth=2.0, label=f'Modelo FOPDT ($R^2 = {r2:.4f}$)')
plt.axvline(t_step / 60.0, color='#3182CE', linestyle='--', alpha=0.8, label=f'Escalón de Potencia ($u = {delta_u}\%$)')
plt.title('Identificación Paramétrica FOPDT - Curva de Reacción Térmica (Invernadero La Paz)', fontweight='bold')
plt.xlabel('Tiempo [minutos]')
plt.ylabel('Temperatura del Aire [°C]')
plt.legend(loc='lower right', framealpha=0.9)
plt.tight_layout()
plt.savefig('fig_fopdt_ajuste_escalon.png', dpi=300)
plt.show()

## 3. Sintonización Clásica de Ganancias PID

Calculamos las ganancias con tres metodologías de referencia en control de procesos:

| Método | $K_p$ | $T_i$ | $T_d$ |
| :--- | :---: | :---: | :---: |
| **Ziegler-Nichols** | $\frac{1.2 \tau}{K \theta}$ | $2 \theta$ | $0.5 \theta$ |
| **Cohen-Coon** | $\frac{1}{K R} \left( \frac{4}{3} + \frac{R}{4} \right)$ | $\theta \left( \frac{32 + 6R}{13 + 8R} \right)$ | $\theta \left( \frac{4}{11 + 2R} \right)$ |
| **IMC** | $\frac{\tau + 0.5\theta}{K(\lambda + 0.5\theta)}$ | $\tau + 0.5\theta$ | $\frac{\tau \theta}{2\tau + \theta}$ |

donde $R = \frac{\theta}{\tau}$ y $\lambda = \max(0.15\tau, 0.8\theta)$.

In [ ]:
# ==============================================================================
# Celda 6: Cálculo de Ganancias PID Clásicas
# ==============================================================================
R = theta_opt / tau_opt

# 1. Ziegler-Nichols
kp_zn = (1.2 * tau_opt) / (K_opt * theta_opt)
ti_zn = 2.0 * theta_opt
td_zn = 0.5 * theta_opt

# 2. Cohen-Coon
kp_cc = (1.0 / (K_opt * R)) * ((4.0 / 3.0) + (R / 4.0))
ti_cc = theta_opt * ((32.0 + 6.0 * R) / (13.0 + 8.0 * R))
td_cc = theta_opt * (4.0 / (11.0 + 2.0 * R))

# 3. IMC
lambda_filtro = max(0.15 * tau_opt, 0.8 * theta_opt)
kp_imc = (tau_opt + 0.5 * theta_opt) / (K_opt * (lambda_filtro + 0.5 * theta_opt))
ti_imc = tau_opt + 0.5 * theta_opt
td_imc = (tau_opt * theta_opt) / (2.0 * tau_opt + theta_opt)

df_ganancias = pd.DataFrame([
    {'Método': 'Ziegler-Nichols', 'Kp': kp_zn, 'Ki': kp_zn/ti_zn, 'Kd': kp_zn*td_zn, 'Ti [s]': ti_zn, 'Td [s]': td_zn},
    {'Método': 'Cohen-Coon', 'Kp': kp_cc, 'Ki': kp_cc/ti_cc, 'Kd': kp_cc*td_cc, 'Ti [s]': ti_cc, 'Td [s]': td_cc},
    {'Método': 'IMC (Modelo Interno)', 'Kp': kp_imc, 'Ki': kp_imc/ti_imc, 'Kd': kp_imc*td_imc, 'Ti [s]': ti_imc, 'Td [s]': td_imc}
])
print(df_ganancias.to_string(index=False))

## 4. Simulación Dinámica en Lazo Cerrado y Métricas de Desempeño

Simulamos el sistema con:
- Setpoint $T_{sp} = 20.0^\circ\text{C}$ (temperatura óptima para desarrollo foliar de papa).
- Perturbación de carga a $t = 1400\text{ s}$ (caída de $-2.5^\circ\text{C}$ por enfriamiento nocturno).
- Anti-windup con integración condicional (*clamping*).
- Derivada sobre la variable de proceso (*derivative on measurement*) para suprimir el *derivative kick*.

In [ ]:
# ==============================================================================
# Celda 7: Función de Simulación en Lazo Cerrado y Métricas
# ==============================================================================
def simular_pid(Kp, Ki, Kd, K, tau, theta, SP=20.0, T0=8.5, t_sim=2400.0, dt=1.0):
    n = int(t_sim / dt)
    t = np.arange(n) * dt
    pasos_retardo = int(round(theta / dt))
    buffer_u = [0.0] * max(1, pasos_retardo)
    
    y = np.zeros(n)
    u = np.zeros(n)
    e_arr = np.zeros(n)
    y[0] = T0
    error_acumulado = 0.0
    pv_prev = T0
    
    for i in range(1, n):
        e = SP - y[i-1]
        e_arr[i] = e
        
        # PID con anti-windup condicional
        p_term = Kp * e
        d_term = -Kd * (y[i-1] - pv_prev) / dt
        pv_prev = y[i-1]
        
        u_tentativa = p_term + error_acumulado + d_term
        if 0.0 <= u_tentativa <= 100.0:
            error_acumulado += Ki * e * dt
            
        u_val = np.clip(p_term + error_acumulado + d_term, 0.0, 100.0)
        u[i] = u_val
        
        buffer_u.append(u_val)
        u_efectiva = buffer_u.pop(0)
        
        pert = -2.5 if t[i] >= 1400.0 else 0.0
        dy = ( - (y[i-1] - (T0 + pert)) + (K * u_efectiva) ) / tau * dt
        y[i] = y[i-1] + dy
        
    return t, y, u, e_arr

def calcular_metricas(t, y, u, e_arr, SP=20.0, T0=8.5):
    dt = t[1] - t[0]
    delta = SP - T0
    Mp = max(0.0, ((np.max(y) - SP) / delta) * 100.0)
    
    idx_10 = np.where(y >= (T0 + 0.10 * delta))[0]
    idx_90 = np.where(y >= (T0 + 0.90 * delta))[0]
    tr = (t[idx_90[0]] - t[idx_10[0]]) if len(idx_10) > 0 and len(idx_90) > 0 else np.nan
    
    idx_pre = np.where(t < 1400.0)[0]
    fuera = np.where((y[idx_pre] < (SP - 0.02 * delta)) | (y[idx_pre] > (SP + 0.02 * delta)))[0]
    ts = t[fuera[-1]] if len(fuera) > 0 else 0.0
    
    ess = abs(SP - y[int(1350/dt)])
    iae = np.sum(np.abs(e_arr[idx_pre])) * dt
    itae = np.sum(t[idx_pre] * np.abs(e_arr[idx_pre])) * dt
    
    return {'tr_s': tr, 'ts_s': ts, 'Mp_pct': Mp, 'ess_c': ess, 'IAE': iae, 'ITAE': itae}

## 5. Optimización de Ganancias con Optuna

Utilizamos **Optuna** con el algoritmo **Tree-structured Parzen Estimator (TPE)** para encontrar las ganancias óptimas que minimizan el error integral ($IAE$) penalizando rigurosamente sobrepasos superiores al $5\%$ para proteger el cultivo de papa.

In [ ]:
# ==============================================================================
# Celda 8: Optimización Bayesiana de Ganancias PID
# ==============================================================================
def objetivo_optuna(trial):
    kp_c = trial.suggest_float('Kp', 1.0, 16.0)
    ki_c = trial.suggest_float('Ki', 0.005, 0.15, log=True)
    kd_c = trial.suggest_float('Kd', 0.1, 5.0)
    
    t, y, u, e_arr = simular_pid(kp_c, ki_c, kd_c, K_opt, tau_opt, theta_opt)
    m = calcular_metricas(t, y, u, e_arr)
    
    costo = m['IAE'] + 50.0 * (max(0.0, m['Mp_pct'] - 5.0)**2)
    return costo

estudio = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
estudio.optimize(objetivo_optuna, n_trials=80)

best_p = estudio.best_params
print("[✔] Optimización completada con éxito.")
print(f"Mejores Ganancias -> Kp: {best_p['Kp']:.3f}, Ki: {best_p['Ki']:.5f}, Kd: {best_p['Kd']:.3f}")

## 6. Comparativa Crítica: Control On/Off vs PID Óptimo

Demostramos la superioridad del control PID continuo frente al termostato On/Off tradicional con histéresis de $\pm 0.5^\circ\text{C}$.

In [ ]:
# ==============================================================================
# Celda 9: Simulación On/Off con Histéresis y Comparación Gráfica
# ==============================================================================
def simular_onoff(K, tau, theta, SP=20.0, T0=8.5, histeresis=0.5, t_sim=2400.0, dt=1.0):
    n = int(t_sim / dt)
    t = np.arange(n) * dt
    buffer_u = [0.0] * max(1, int(round(theta/dt)))
    y = np.zeros(n)
    u = np.zeros(n)
    e_arr = np.zeros(n)
    y[0] = T0
    estado = 0.0
    conmutaciones = 0
    
    for i in range(1, n):
        e_arr[i] = SP - y[i-1]
        if y[i-1] < (SP - histeresis) and estado == 0.0:
            estado = 100.0
            conmutaciones += 1
        elif y[i-1] > (SP + histeresis) and estado == 100.0:
            estado = 0.0
            conmutaciones += 1
        u[i] = estado
        buffer_u.append(estado)
        u_ef = buffer_u.pop(0)
        pert = -2.5 if t[i] >= 1400.0 else 0.0
        y[i] = y[i-1] + (( - (y[i-1] - (T0 + pert)) + K * u_ef ) / tau * dt)
    return t, y, u, e_arr, conmutaciones

t_on, y_on, u_on, e_on, conmut = simular_onoff(K_opt, tau_opt, theta_opt)
t_pid, y_pid, u_pid, e_pid = simular_pid(best_p['Kp'], best_p['Ki'], best_p['Kd'], K_opt, tau_opt, theta_opt)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6.5), dpi=300, sharex=True)
ax1.plot(t_on/60.0, y_on, label='Control On/Off con Histéresis (±0.5 °C)', color='#E53E3E', alpha=0.8)
ax1.plot(t_pid/60.0, y_pid, label='Control PID Óptimo (Optuna)', color='#38A169', linewidth=2.0)
ax1.axhline(20.0, color='#1A202C', linestyle=':', label='Setpoint (20.0 °C)')
ax1.set_ylabel('Temperatura [°C]')
ax1.set_title('Comparativa de Regulación Térmica: Control On/Off vs PID Óptimo', fontweight='bold')
ax1.legend(loc='lower right')

ax2.plot(t_on/60.0, u_on, label=f'Actuador On/Off ({conmut} conmutaciones)', color='#E53E3E', alpha=0.6)
ax2.plot(t_pid/60.0, u_pid, label='Actuador PID (Modulación Suave SSR)', color='#38A169', linewidth=1.8)
ax2.set_ylabel('Esfuerzo de Control [%]')
ax2.set_xlabel('Tiempo [minutos]')
ax2.set_ylim(-5, 105)
ax2.legend(loc='upper right')
plt.tight_layout()
plt.savefig('fig_comparativa_onoff_vs_pid.png', dpi=300)
plt.show()

## 7. Tabla Consolidada de Métricas para la Memoria de Tesis
Generamos la tabla comparativa final con todos los controladores evaluados:

In [ ]:
# ==============================================================================
# Celda 10: Consolidación de Métricas en Tabla Comparativa
# ==============================================================================
met_zn = calcular_metricas(*simular_pid(kp_zn, kp_zn/ti_zn, kp_zn*td_zn, K_opt, tau_opt, theta_opt))
met_cc = calcular_metricas(*simular_pid(kp_cc, kp_cc/ti_cc, kp_cc*td_cc, K_opt, tau_opt, theta_opt))
met_imc = calcular_metricas(*simular_pid(kp_imc, kp_imc/ti_imc, kp_imc*td_imc, K_opt, tau_opt, theta_opt))
met_opt = calcular_metricas(t_pid, y_pid, u_pid, e_pid)
met_on = calcular_metricas(t_on, y_on, u_on, e_on)

tabla_final = pd.DataFrame([
    {'Método': 'Ziegler-Nichols', 'Kp': f'{kp_zn:.2f}', 'Ki': f'{kp_zn/ti_zn:.4f}', 'Kd': f'{kp_zn*td_zn:.2f}', 'tr [s]': f"{met_zn['tr_s']:.1f}", 'ts [s]': f"{met_zn['ts_s']:.1f}", 'Mp [%]': f"{met_zn['Mp_pct']:.1f}", 'IAE': f"{met_zn['IAE']:.1f}"},
    {'Método': 'Cohen-Coon', 'Kp': f'{kp_cc:.2f}', 'Ki': f'{kp_cc/ti_cc:.4f}', 'Kd': f'{kp_cc*td_cc:.2f}', 'tr [s]': f"{met_cc['tr_s']:.1f}", 'ts [s]': f"{met_cc['ts_s']:.1f}", 'Mp [%]': f"{met_cc['Mp_pct']:.1f}", 'IAE': f"{met_cc['IAE']:.1f}"},
    {'Método': 'IMC (Modelo Interno)', 'Kp': f'{kp_imc:.2f}', 'Ki': f'{kp_imc/ti_imc:.4f}', 'Kd': f'{kp_imc*td_imc:.2f}', 'tr [s]': f"{met_imc['tr_s']:.1f}", 'ts [s]': f"{met_imc['ts_s']:.1f}", 'Mp [%]': f"{met_imc['Mp_pct']:.1f}", 'IAE': f"{met_imc['IAE']:.1f}"},
    {'Método': 'Optuna (Óptimo)', 'Kp': f"{best_p['Kp']:.2f}", 'Ki': f"{best_p['Ki']:.4f}", 'Kd': f"{best_p['Kd']:.2f}", 'tr [s]': f"{met_opt['tr_s']:.1f}", 'ts [s]': f"{met_opt['ts_s']:.1f}", 'Mp [%]': f"{met_opt['Mp_pct']:.1f}", 'IAE': f"{met_opt['IAE']:.1f}"},
    {'Método': 'Control On/Off (±0.5°C)', 'Kp': 'N/A', 'Ki': 'N/A', 'Kd': 'N/A', 'tr [s]': f"{met_on['tr_s']:.1f}", 'ts [s]': 'Oscila', 'Mp [%]': f"{met_on['Mp_pct']:.1f}", 'IAE': f"{met_on['IAE']:.1f}"}
])

print(tabla_final.to_string(index=False))
tabla_final.to_csv('tabla_comparativa_controladores.csv', index=False)